# 07 - Biaya indeks FAISS dan arsip hasil

Dijalankan paling akhir, setelah `06_artifacts.ipynb`, supaya gambar dan data tabel ikut terbungkus di arsip. Dua hal: biaya indeks FAISS lintas k, dan arsip `hasil_*.tar.gz` untuk diunduh sebelum instance dihancurkan.

In [ ]:
from pathlib import Path

# Folder hasil yang diarsipkan; arsip ditulis ke root repo, di luar folder ini
OUTPUT_DIR = Path("../outputs")
ARCHIVE_DIR = Path("..")
OUT_DIR = OUTPUT_DIR / "tuning"
MODEL_NAME = "indobenchmark/indobert-base-p2"
FEATURE_DIR = OUT_DIR / "features" / MODEL_NAME.replace("/", "__")
# Keluaran pengukuran biaya indeks FAISS
FAISS_DIR = OUTPUT_DIR / "faiss_index"
# Nilai k yang diukur dan jumlah pengulangan tiap pengukuran waktu
K_VALUES = [1, 3, 5, 10, 20, 50]
REPEATS = 20
# Ikutkan checkpoints/ dan features/ di arsip (besar: checkpoint RM-a sekitar 420 MB per berkas)
INCLUDE_HEAVY = False

## 1. Biaya indeks FAISS

RM-c tidak melatih apa pun (biaya latihnya adalah biaya head RM-b yang dipakainya), tetapi menambah biaya retrieval saat inferensi. Biayanya terbagi dua: pembangunan indeks sekali dari embedding train, serta penelusuran pada setiap inferensi yang tumbuh mengikuti k. Query memakai embedding test; indeks tetap hanya dari train.

In [ ]:
import json
import time

import faiss
import numpy as np
import pandas as pd
import psutil
from IPython.display import display


def load_json(path: Path, default: dict | None = None) -> dict | None:
    """Baca berkas JSON; `default` bila berkas belum ada."""
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding="utf-8"))


def load_features(feature_dir: Path) -> dict:
    """Baca embedding beku dan label ketiga split dari cache 03b.

    Raises:
        FileNotFoundError: Bila cache belum ada.
    """
    if not (feature_dir / "extract_meta.json").exists():
        raise FileNotFoundError(f"Cache fitur tidak ada di {feature_dir}; jalankan 03b_rmb_frozen lebih dulu")
    meta = load_json(feature_dir / "extract_meta.json")
    return {
        "embeddings": {split: np.load(feature_dir / f"{split}_emb.npy") for split in ["train", "val", "test"]},
        "labels": {split: np.load(feature_dir / f"{split}_label.npy") for split in ["train", "val", "test"]},
        "extract_time_s": float(meta["extract_time_s"]),
        "extract_peak_mem_mb": float(meta["extract_peak_gpu_mem_mb"]),
    }


def l2_normalize(matrix: np.ndarray) -> np.ndarray:
    """Normalisasi L2 per baris agar inner product setara cosine."""
    matrix = np.ascontiguousarray(matrix, dtype=np.float32)
    return matrix / np.clip(np.linalg.norm(matrix, axis=1, keepdims=True), 1e-12, None)


def measure_index_build(train_embeddings: np.ndarray, repeats: int) -> dict:
    """Waktu bangun indeks flat (rata-rata dan minimum), ukuran indeks, dan kenaikan RSS proses."""
    normalized = l2_normalize(train_embeddings)
    n_vectors, dim = normalized.shape
    process = psutil.Process()
    rss_before = process.memory_info().rss
    durations = []
    for _ in range(repeats):
        started = time.perf_counter()
        index = faiss.IndexFlatIP(dim)
        index.add(normalized)
        durations.append(time.perf_counter() - started)
    rss_after = process.memory_info().rss
    return {
        "n_vectors": int(n_vectors), "dim": int(dim),
        "build_time_ms_mean": round(float(np.mean(durations)) * 1000, 4),
        "build_time_ms_min": round(float(np.min(durations)) * 1000, 4),
        # Indeks flat menyimpan seluruh vektor apa adanya sebagai float32 (4 byte)
        "index_size_mb": round(n_vectors * dim * 4 / 1024**2, 3),
        "rss_delta_mb": round((rss_after - rss_before) / 1024**2, 2),
        "index_ntotal": int(index.ntotal),
    }


def measure_search(train_embeddings: np.ndarray, query_embeddings: np.ndarray, k_values: list[int], repeats: int) -> pd.DataFrame:
    """Waktu penelusuran satu batch query dan per query untuk tiap k."""
    normalized_train = l2_normalize(train_embeddings)
    index = faiss.IndexFlatIP(normalized_train.shape[1])
    index.add(normalized_train)
    normalized_query = l2_normalize(query_embeddings)
    rows = []
    for k in k_values:
        # Satu penelusuran tanpa dihitung: tanpa ini k pertama menanggung biaya cache dingin dan start thread OpenMP
        index.search(normalized_query, k)
        durations = []
        for _ in range(repeats):
            started = time.perf_counter()
            index.search(normalized_query, k)
            durations.append(time.perf_counter() - started)
        mean_seconds = float(np.mean(durations))
        rows.append({"k": int(k), "n_queries": len(normalized_query),
                     "search_time_ms_batch": round(mean_seconds * 1000, 4),
                     "search_time_us_per_query": round(mean_seconds / len(normalized_query) * 1_000_000, 4)})
    return pd.DataFrame(rows)


features = load_features(FEATURE_DIR)
build = measure_index_build(features["embeddings"]["train"], REPEATS)
search = measure_search(features["embeddings"]["train"], features["embeddings"]["test"], K_VALUES, REPEATS)

FAISS_DIR.mkdir(parents=True, exist_ok=True)
(FAISS_DIR / "faiss_build.json").write_text(json.dumps(build, ensure_ascii=False, indent=2), encoding="utf-8", newline="")
search.to_csv(FAISS_DIR / "faiss_search.csv", index=False, lineterminator="\n")
print(pd.Series(build).to_string())
display(search)

Indeks bertipe flat/exact, jadi penelusuran adalah brute force atas seluruh vektor train. Untuk ukuran data ini biayanya kecil dan hasilnya deterministik, yang jauh lebih penting untuk penelitian daripada penghematan waktu dari indeks aproksimasi.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(search["k"], search["search_time_us_per_query"], marker="o")
ax.set_xlabel("k (jumlah tetangga)")
ax.set_ylabel("mikrodetik per query")
ax.set_title("Biaya penelusuran indeks FAISS")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 2. Arsipkan hasil

`outputs/` tidak disimpan di git, jadi hasil kampanye hanya ada di disk mesin tempat notebook ini berjalan dan hilang bersama instance yang dihancurkan. Angka efisiensi (waktu latih, latency, memori) tidak bisa dibuat ulang karena hanya sah dari sesi aslinya. Sel di bawah membungkus `outputs/` menjadi satu `hasil_<gpu>_<waktu>.tar.gz` di root repo; unduh sebelum instance dihancurkan.

`checkpoints/` dan `features/` tidak ikut kecuali `INCLUDE_HEAVY = True`. Tanpa keduanya, 05 tidak bisa dijalankan ulang tanpa mengulang tuning.

In [ ]:
import os
import re
import tarfile

# Karakter selain huruf kecil dan angka pada nama GPU, contoh: "NVIDIA GeForce RTX 3090" -> "nvidia-geforce-rtx-3090"
NON_SLUG_PATTERN = re.compile(r"[^a-z0-9]+")


def format_hardware_label(output_dir: Path) -> str:
    """Nama GPU dari `hardware.json` pertama di bawah `output_dir`, sebagai slug."""
    for path in sorted(output_dir.glob("*/hardware.json")):
        slug = NON_SLUG_PATTERN.sub("-", str(load_json(path, default={}).get("gpu", "")).lower()).strip("-")
        if slug:
            return slug
    return "tanpa-hardware"


def list_archive_files(output_dir: Path, include_heavy: bool) -> list[Path]:
    """Seluruh berkas di bawah `output_dir`, tanpa `checkpoints/` dan `features/` bila tidak diminta."""
    excluded = set() if include_heavy else {"checkpoints", "features"}
    files = []
    for root, dirs, names in os.walk(output_dir):
        dirs[:] = sorted(name for name in dirs if name not in excluded)
        files.extend(Path(root) / name for name in sorted(names))
    return files


def save_archive(output_dir: Path, archive_dir: Path, include_heavy: bool) -> Path:
    """Tulis `hasil_<gpu>_<waktu>.tar.gz`; berkas sementara `.tmp` dihapus bila gagal.

    Raises:
        FileNotFoundError: Bila tidak ada berkas hasil.
    """
    output_dir = output_dir.resolve()
    files = list_archive_files(output_dir, include_heavy)
    if not files:
        raise FileNotFoundError(f"Tidak ada hasil untuk diarsipkan di {output_dir}")
    # Contoh: hasil_nvidia-geforce-rtx-3090_20260920-143005.tar.gz
    target = archive_dir.resolve() / f"hasil_{format_hardware_label(output_dir)}_{time.strftime('%Y%m%d-%H%M%S')}.tar.gz"
    partial = target.with_name(target.name + ".tmp")
    try:
        with tarfile.open(partial, "w:gz") as archive:
            for path in files:
                archive.add(path, arcname=(Path(output_dir.name) / path.relative_to(output_dir)).as_posix(), recursive=False)
        os.replace(partial, target)
    except OSError:
        partial.unlink(missing_ok=True)
        raise
    return target


archive_path = save_archive(OUTPUT_DIR, ARCHIVE_DIR, INCLUDE_HEAVY)
print(f"arsip  : {archive_path}")
print(f"ukuran : {archive_path.stat().st_size / 1024**2:.1f} MB")

## Ringkasan

Bahan penulisan lengkap: tabel metrik di `outputs/tuning/metrics/`, gambar dan data tabel di `outputs/tuning/artifacts/`, dan biaya retrieval di `outputs/faiss_index/`. Semuanya ikut dalam arsip `hasil_*.tar.gz` di root repo; unduh sebelum instance dihancurkan.